In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import logging
from utils.observability import create_control_table, insert_control_record
from utils.shared import load_table, notebook_exit
from logger.custom_logging import get_job_logger, set_up_logger
from schema.silver import explode_conn_schema, conn_table_schema
from pyspark.sql.functions import max
from utils.silver import (
    add_columns_for_idempotency,
    add_scd_logic,
    explode_connections,
    select_conn_columns,
    identify_rows_to_expire,
    insert_new_rows,
    join_current_and_incoming_data,
    load_current_data,
    update_expired_rows,
    validate_columns,
    validate_and_quarantine_rows,
)

try:
    context = REPLACED_WITH_SPARK_CONF
    run_id = context.tags().get("runId").get()
except:
    # Fallback for when running as a file (not a notebook)
    run_id = "unknown"

log_info = {
    "layer": "silver",
    "job": "process_station_connector_data",
    "dataset": "bronze_dev.electrovolt.ev_stations",
}

logger = set_up_logger()

log = get_job_logger(logger, **log_info, run_id=run_id)

log(logging.INFO, "Starting process_station_connector_data")

# The variables below are for the control table which is used to track the last processed timestamp and observe run metrics

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
pipeline = "project_voltstream"
layer = f"{log_info["layer"]}.{log_info["job"]}"
error_message = None
records_processed = 0
records_failed = 0
last_processed_timestamp = None

try:
    control_table = create_control_table(spark)
    df = spark.table("silver_dev.electrovolt.valid_stations")
    df = explode_connections(df, **log_info)
    df = select_conn_columns(df)
    df = validate_columns(df, **log_info)
    df, df_quarantine = validate_and_quarantine_rows(
        df,
        ["station_id", "connection_type_id", "level_id", "quantity", "power_kw"],
        ["quantity", "power_kw"],
        **log_info,
    )
    records_processed = df.count()
    records_failed = df_quarantine.count()

    conn_df = spark.createDataFrame([], schema=conn_table_schema)
    conn_table = load_table(spark, "silver_dev.electrovolt.ev_connections", conn_df)

    table_df = spark.table("silver_dev.electrovolt.ev_connections")
    current_conn_dim = load_current_data(table_df, **log_info)

    joint_conn_df = join_current_and_incoming_data(
        df, current_conn_dim, ["station_id", "connection_type_id", "level_id"]
    )

    joint_conn_df = add_scd_logic(
        joint_conn_df,
        "connection_sk",
        ["amps", "voltage", "power_kw", "quantity", "current_type"],
    )

    inserts_conn_df = add_columns_for_idempotency(
        joint_conn_df,
        ["station_id", "connection_type_id", "level_id", "date_last_status_update"],
        "connection_sk",
        [
            "connection_type_id",
            "connection_type",
            "station_id",
            "level_id",
            "amps",
            "voltage",
            "power_kw",
            "quantity",
            "current_type",
            "ingest_timestamp",
        ],
    )

    expired_conn_df = identify_rows_to_expire(joint_conn_df, "connection_sk")

    updated_conn_df = update_expired_rows(
        conn_table, expired_conn_df, "connection_sk", **log_info
    )

    final_conn_df = insert_new_rows(
        conn_table, inserts_conn_df, "connection_sk", **log_info
    )
    status = "success"
    last_processed_timestamp = df.agg(max("ingest_timestamp")).collect()[0][0]
    log(logging.INFO, "Finished process_station_connector_data")

except Exception as e:
    status = "failed"
    error_message = str(e)
    raise

end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
insert_control_record(
    spark,
    control_table,
    pipeline,
    layer,
    last_processed_timestamp,
    records_processed,
    records_failed,
    start_time,
    end_time,
    error_message,
    status,
)
notebook_exit("SUCCESS")